## 1 Introduction


*Team members*
- Alexandre Dréan (2408681)
- Julien Segonne (2409827)

In [ ]:
import pandas as pd
from scipy.io import mmread
import scipy.sparse as sp
import torch
import torch.nn as nn
import scipy.sparse as sp
import numpy as np

In [ ]:
ids = pd.read_csv('data/TP4-ids.csv', index_col=0)
articles = pd.concat([
    pd.read_csv('data/TP4-articles1.csv', index_col=0),
    pd.read_csv('data/TP4-articles2.csv', index_col=0)
], ignore_index=True)

adjacency = mmread('data/TP4-matrice-adjacence.dgt').tocsr() # Shape : (67294, 67294) (only save non-zero edges)

In [16]:
# This is a way to find the articles cited by an article with a given index (here n=1)

n = 1
i = n - 1

cited_indices = adjacency[i].indices
cited_n = cited_indices + 1



cited_ids = ids[ids['n'].isin(cited_n)]['id'].values
cited_articles = articles[articles['id'].isin(cited_ids)][['id', 'title', 'year']]

cited_articles


,id,title,year
1,573695ff6e3b12023e513394,Exploring the Limits of Language Modeling.,2016
2,56d81592dabfae2eee6f4c65,A Hierarchical Pitman−Yor Process HMM for Unsu...,2011
3,53e9ad2db7602d97037103c6,Painless unsupervised learning with features,2010
4,53e9bd8cb7602d9704a36c0b,Unsupervised part-of-speech tagging with bilin...,2011
5,573696da6e3b12023e5d96a9,Labeled Grammar Induction with Minimal Supervi...,2015
6,5736960f6e3b12023e521c63,Recurrent Memory Network for Language Modeling,2016
7,5550401245ce0a409eb3205c,Dropout: a simple way to prevent neural networ...,2014
8,53e9982cb7602d970204e432,Phylogenetic Grammar Induction,2010
9,53e9a645b7602d9702f7362e,Noise-contrastive estimation: A new estimation...,2010
10,573695fd6e3b12023e51147e,Variational Dropout and the Local Reparameteri...,2015


## 2 PageRank algorithm

### 2.1 What is PageRank algorithm and why using it ?
TODO

### 2.2 PageRank code

In [17]:
# TODO

### 2.3 Performances

In [18]:
# TODO

### 2.4 Discussion

TODO

## 3 NGCF algorithm

### 3.1 What is NGCF algorithm and why using it ?

L'algorithme [Neural Graph Collaborative Filtering](./https://arxiv.org/pdf/1905.08108) à pour idée clé de partir du contenu d'un item et d'en faire un embedding, puis cet embedding est enrichi itérativement en parcourant le graphe de relation à partir de cet item. Les informations sémantique contenu dans les relations entre l'item et ces voisins viennent donc enrichir l'information encodé dans l'embedding associé à cet item. 

Cela est particulièrement pertinent dans le cadre de cette étude, en effet nous disposons d'informations pertinentes sur les articles en terme de contenu pur (titre, auteurs, abstract), mais le graphe de citation contient aussi des informations supplémentaires pertinentes, étudier les relations connus entre les articles pourrait par exemple permettre d'apprendre que certains auteurs se citent souvent entre sans pour autant être les auteurs avec les publications les plus proches, ce qui n'aurait pas été possible avec le contenu seul.

### 3.2 NGCF Code

In [ ]:
class NGCF_ItemItem(nn.Module):
    def __init__(self, E0, adjacency, n_layers=3, embed_size=512):
        super().__init__()
        n_items = E0.shape[0]
        
        # Embedding initial (depuis CLIP/SBERT)
        self.E0 = nn.Parameter(torch.tensor(E0, dtype=torch.float32))
        
        # Matrices W pour chaque couche
        self.W1 = nn.ModuleList([nn.Linear(embed_size, embed_size, bias=False) 
                                  for _ in range(n_layers)])
        self.W2 = nn.ModuleList([nn.Linear(embed_size, embed_size, bias=False) 
                                  for _ in range(n_layers)])
        
        self.L = self.normalize(adjacency)  # (L = D^-1/2 · A · D^-1/2)
        self.n_layers = n_layers

    def normalize(self, A):
        degrees = np.array(A.sum(axis=1)).flatten()
        D_inv_sqrt = sp.diags(1.0 / np.sqrt(degrees + 1e-8))
        L = D_inv_sqrt @ A @ D_inv_sqrt

        # on garde la matrice creuse pour les perfs
        L_coo = L.tocoo()
        indices = torch.tensor([L_coo.row, L_coo.col], dtype=torch.long)
        values = torch.tensor(L_coo.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices, values, L.shape)

    def forward(self):


        E = self.E0
        all_embeddings = [E]
        
        # on propage les embeddings à travers le graphe
        for l in range(self.n_layers):
            # (L + I)E W1  +  L(E ⊙ E) W2
            LE = torch.sparse.mm(self.L, E)
            term1 = self.W1[l](LE + E)           # (L+I)E · W1
            term2 = self.W2[l](LE * E)            # L(E⊙E) · W2
            E = nn.functional.leaky_relu(term1 + term2)
            all_embeddings.append(E)
        
        # Concatenation de toutes les couches
        E_star = torch.cat(all_embeddings, dim=1)
        return E_star

    def bpr_loss(self, E_star, i, j, k):
        # i cite j (positif), i ne cite pas k (négatif)
        e_i = E_star[i]
        e_j = E_star[j]
        e_k = E_star[k]
        
        pos_score = (e_i * e_j).sum(dim=1)
        neg_score = (e_i * e_k).sum(dim=1)
        
        loss = -torch.log(torch.sigmoid(pos_score - neg_score)).mean()
        return loss

## 4 Third algorithm

## 5 Fourth algorithm